### 1. Project Overview

#### Objective

Build a multi-source Delta Live Tables (DLT) pipeline using Bronze, Silver, and Gold architecture.

Pipeline Flow

Users CSV + Orders CSV
        ↓
     Bronze
        ↓
     Silver
        ↓
      Gold

### 2. Imports & Setup

#### Import Required Libraries

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

### 3. Create Catalog & Schema

#### Create database

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS ecommerce_dlt_db;

#### Use database

In [0]:
%sql
USE workspace.ecommerce_dlt_db;

### 4. Load Users Dataset

#### Define Users Dataset Path

In [0]:
users_path = "/Volumes/workspace/ecommerce_dlt_db/source_files/users.csv"

### 5. Load Orders Dataset

#### Define Orders Dataset Path

In [0]:
orders_path = "/Volumes/workspace/ecommerce_dlt_db/source_files/orders.csv"

### 6. Bronze Layer

#### Bronze Users Table

In [0]:
@dp.table(
    name = "users_bronze",
    comment = "Raw data from users table"
)

def users_bronze():
    return(
        spark.read.format("csv")
            .option("header", "true")
                .load(users_path)
    )






#### Bronze Orders Table

In [0]:
@dp.table(
    name = "orders_bronze",
    comment = "Raw data from orders table"
)

def orders_bronze():
    return(
        spark.read.format("csv")
            .option("header", "true")
                .load(orders_path)
    )

### 7. Silver Layer

#### Silver Users Table

In [0]:
@dp.table(
    name = "silver_users",
    comment = "cleaned silver table for users"
)

@dp.expect_or_drop(
    "valid user Id",
    "user_id is not NULL"
)

def silver_users():
    return(
        dp.read("users_bronze")
            .dropDuplicates()
    )

#### Silver Orders Table

In [0]:
@dp.table(
    name = "silver_orders",
    comment = "cleaned orders table"
)

@dp.expect_or_drop(
    "valid amount",
    "amount > 0"
)

def silver_orders():
    return(
        dp.read("orders_bronze")
            .withColumn("amount", col("amount").cast("int"))
            .dropDuplicates()
    )

### 8. Join Transformations

#### Create Joined Customer Orders Table

In [0]:
@dp.table(
    name = "silver_customers_orders",
    comment = "Joined users and orders dataset"
)

def silver_customers_orders():
    users_df = dp.read("silver_users")
    orders_df = dp.read("silver_orders")
    return(
        users_df.join(orders_df, on = "user_id", how = "inner")
    )

### 9. Gold Layer Aggregations

#### Customer Spending Analytics

In [0]:
@dp.table(
    name = "gold_customer_spending",
    comment = "customer_spending_analytics"
)

def gold_customer_spending():
    df = dp.read("silver_customers_orders")

    return(
        df.groupBy("user_id", "name")\
            .agg(
                count("order_id").alias("total_orders"),
                sum("amount").alias("total_spent")
            )
    )

### 10. DLT Expectations

#### Expectations Used

1. user_id should not be null

2. amount should be greater than 0

3. duplicate records are removed in silver layer

### 11. Pipeline Execution

#### Steps to Execute Pipeline

1. Go to Jobs & Pipelines

2. Create ETL Pipeline

3. Select this notebook as source

4. Choose target schema

5. Run pipeline

6. Observe DAG and lineage